# Phase 4 Continuous Wrist Kinematics Estimation

This notebook presents the organized implementation of the submitted bachelor’s degree project. It preserves the executed Phase 4 parameters, data transformations, CNN-LSTM architecture, training callbacks, Gaussian output smoothing, gain calibration, and sample export.

The full translated research report is in `../report/`. Read `../TECHNICAL_NOTES.md` for the distinction between surrogate targets and measured angles, and for the original validation and offline-processing limitations. The original execution log is preserved separately in `../results/original_training_log.txt`. The code cells below have not been rerun against the missing raw dataset.


In [ ]:
from pathlib import Path
import sys

# Locate the project whether Jupyter starts in the root or notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src/wrist_kinematics").is_dir():
    raise FileNotFoundError("Start this notebook from the project or notebooks folder.")
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
from wrist_kinematics.config import ExperimentConfig
from wrist_kinematics.evaluation import (
    calculate_metrics, plot_predictions, postprocess_predictions, save_metrics,
)


## Configuration

The input is `data/DB2_s1/S1_E1_A1.mat` by default. It must contain `emg`, `restimulus`, and `rerepetition`. The source uses subject 1, 600-sample windows, a step of 100 samples, a sampling rate of 2,000 Hz, and a 5 Hz low-pass filter. Repetitions 1, 3, 4, and 6 are used for training; repetitions 2 and 5 are used for validation and evaluation, as in the original implementation.


In [ ]:
config = ExperimentConfig(data_dir=PROJECT_ROOT / "data")
run_dir = PROJECT_ROOT / "results/notebook_run"
run_dir.mkdir(parents=True, exist_ok=True)
config


## Inspect the Preserved Samples Without Retraining

This section can be run independently of the raw dataset and TensorFlow. The saved predictions already include the gain of 1.35 and clipping to [−1, +1]. These recomputed metrics are separate from the pre-calibration values printed in the original notebook and published in the report.


In [ ]:
saved_true = np.load(PROJECT_ROOT / "results/samples/y_true_sample.npy")
saved_pred = np.load(PROJECT_ROOT / "results/samples/y_pred_sample.npy")
saved_metrics = calculate_metrics(saved_true, saved_pred)
saved_metrics


In [ ]:
plot_predictions(saved_true, saved_pred, show=True)


## Preprocess the Raw Recording

The implementation selects rest and movement codes 9, 10, 13, 14, 15, and 16; rectifies the signal; applies the fourth-order Butterworth filter; and fits StandardScaler on training-repetition samples only. Movement codes are mapped to three normalized target axes and smoothed using a Gaussian filter with sigma 400. Windows retain the target and repetition label of their final sample. Filtering, concatenation, window boundaries, and split assignment follow the source exactly.


In [ ]:
from wrist_kinematics.preprocessing import prepare_data

data = prepare_data(config)
print(f"Training shape: {data.X_train.shape}")
print(f"Evaluation shape: {data.X_test.shape}")


## Final CNN-LSTM Architecture

The model contains GaussianNoise(0.05), two Conv1D stages with 32 and 64 filters, BatchNormalization, MaxPooling1D, SpatialDropout1D(0.2), a 64-unit LSTM, Dropout(0.3), GlobalAveragePooling1D, a 64-unit Dense layer, and three linear outputs. L2 regularization is 0.005. The executed training configuration uses Adam with learning rate 0.00025, MSE loss, and MAE monitoring. The initial compile at 0.0005 in the source is superseded by the later compile and is therefore omitted.


In [ ]:
from wrist_kinematics.model import build_model

model = build_model(config)
model.summary()


## Train to a New Checkpoint

Training runs for up to 60 epochs with batch size 64. The callbacks preserve the best validation-loss checkpoint, stop after 15 epochs without improvement, and halve the learning rate after five such epochs, down to a minimum of 0.00001. The supplied trained model remains intact; a new checkpoint is used here.


In [ ]:
import json
from wrist_kinematics.model import train_model

new_checkpoint = PROJECT_ROOT / "models/phase4_notebook_run.keras"
model, history = train_model(data, config, new_checkpoint)
(run_dir / "training_history.json").write_text(
    json.dumps(history.history, indent=2), encoding="utf-8"
)


## Load the Best Model and Predict

Select the preserved submitted checkpoint or the new checkpoint from the preceding section. Model loading uses `compile=False` because evaluation does not require restoring optimizer state.


In [ ]:
from tensorflow.keras.models import load_model

model_path = config.model_path  # Change to new_checkpoint to evaluate the new run.
model = load_model(model_path, compile=False)
raw_predictions = model.predict(data.X_test)
smoothed_predictions, calibrated_predictions = postprocess_predictions(
    raw_predictions, sigma=config.prediction_sigma, gain=config.gain_factor
)


## Evaluate Before and After Calibration

The original notebook prints RMSE and Pearson correlation using the smoothed predictions before calibration. It then multiplies by 1.35, clips to [−1, +1], plots, and saves the calibrated predictions. This notebook preserves that order and exports both sets of metrics to make the distinction explicit.


In [ ]:
pre_calibration_metrics = calculate_metrics(data.y_test, smoothed_predictions)
calibrated_metrics = calculate_metrics(data.y_test, calibrated_predictions)
save_metrics(pre_calibration_metrics, run_dir / "pre_calibration_metrics.csv")
save_metrics(calibrated_metrics, run_dir / "calibrated_metrics.csv")
print("Pre-calibration:", pre_calibration_metrics)
print("Calibrated:", calibrated_metrics)
plot_predictions(
    data.y_test, calibrated_predictions,
    path=run_dir / "predictions.png", num_samples=99000, show=True,
)


## Export the Sample Arrays

The original variable names and NumPy output format are retained, but new outputs are written to a separate run folder.


In [ ]:
np.save(run_dir / "y_true_sample.npy", data.y_test)
np.save(run_dir / "y_pred_sample.npy", calibrated_predictions)
np.save(run_dir / "y_pred_smoothed.npy", smoothed_predictions)


## Replay the 3D Demonstration

Run `python -m wrist_kinematics visualize` from a terminal to replay the supplied arrays. To replay a new run, pass `--samples-dir results/notebook_run`. The display uses the source demonstration’s illustrative ±90° scale rather than a measured physical calibration.


## Supplementary Diagnostics

The expanded report adds residual, scatter, activity-conditioned, zero-baseline, and model-configuration analyses of the supplied arrays. These describe existing exported predictions; they do not claim a new training run.


In [ ]:
from wrist_kinematics.diagnostics import analyze_samples

extended_summary = analyze_samples(
    PROJECT_ROOT / "results/samples",
    PROJECT_ROOT / "results/extended_analysis",
    model_path=PROJECT_ROOT / "models/paper_model_best.keras",
    activity_threshold=0.05,
)
extended_summary["axis_metrics"]


## Raw Recording Audit

After obtaining the missing recording, inspect its original timeline and repetition labels. The audit counts selected-sequence gaps and windows containing more than one repetition identifier. A mixed repetition is a diagnostic flag rather than an automatic proof of cross-partition leakage.


In [ ]:
from wrist_kinematics.diagnostics import inspect_recording

recording_path = config.data_dir / "DB2_s1/S1_E1_A1.mat"
if recording_path.is_file():
    recording_audit = inspect_recording(recording_path)
    print(recording_audit)
else:
    print("Raw recording unavailable; the actual recording audit has not been performed.")
